In [9]:
from pathlib import Path

documents_path = Path("../documents")

for file in documents_path.rglob("*.txt"):
    print(file)

..\documents\logistics\carrier_service_guidelines.txt
..\documents\logistics\incoterms_guidance.txt
..\documents\supplier\supplier_delivery_policy.txt


In [10]:
documents = []

for file in documents_path.rglob("*.txt"):
    text = file.read_text(encoding="utf-8")
    
    documents.append({
        "source": str(file),
        "text": text
    })

print("Number of documents:", len(documents))

Number of documents: 3


In [11]:
print(documents[0]["source"])
print()
print(documents[0]["text"][:1000])

..\documents\logistics\carrier_service_guidelines.txt

Document Type: Carrier Service Guidelines
Document Status: Illustrative project document
Document ID: CAR-001

1. Shipment Planning

Carrier selection should consider the required delivery date, shipment origin, destination, service level, cost, and applicable transport restrictions.

2. Standard Service

Standard service is intended for shipments where the agreed delivery date allows normal transportation lead time.

The logistics team should compare the carrier's expected transit time with the required delivery date before selecting the service.

3. Expedited Service

Expedited service may be considered when the required delivery date cannot reasonably be achieved using standard transportation.

Expedited transportation may result in higher transportation costs.

4. Shipment Tracking

Shipment tracking information should be captured where available.

Important tracking events may include:

* Shipment created
* Shipment picked up


In [12]:
chunks = []

for document in documents:
    text = document["text"]
    source = document["source"]

    paragraphs = [
        paragraph.strip()
        for paragraph in text.split("\n\n")
        if paragraph.strip()
    ]

    for paragraph in paragraphs:
        chunks.append({
            "source": source,
            "text": paragraph
        })

print("Number of chunks:", len(chunks))

Number of chunks: 57


In [13]:
for i, chunk in enumerate(chunks[:5]):
    print(f"--- Chunk {i + 1} ---")
    print("Source:", chunk["source"])
    print(chunk["text"])
    print()

--- Chunk 1 ---
Source: ..\documents\logistics\carrier_service_guidelines.txt
Document Type: Carrier Service Guidelines
Document Status: Illustrative project document
Document ID: CAR-001

--- Chunk 2 ---
Source: ..\documents\logistics\carrier_service_guidelines.txt
1. Shipment Planning

--- Chunk 3 ---
Source: ..\documents\logistics\carrier_service_guidelines.txt
Carrier selection should consider the required delivery date, shipment origin, destination, service level, cost, and applicable transport restrictions.

--- Chunk 4 ---
Source: ..\documents\logistics\carrier_service_guidelines.txt
2. Standard Service

--- Chunk 5 ---
Source: ..\documents\logistics\carrier_service_guidelines.txt
Standard service is intended for shipments where the agreed delivery date allows normal transportation lead time.



In [14]:
import ollama

models = ollama.list()

for model in models["models"]:
    print(model["model"])

nomic-embed-text:latest
llama3.2:latest


In [15]:
import ollama

embedding_response = ollama.embed(
    model="nomic-embed-text",
    input="What information should a supplier provide?"
)

print(type(embedding_response))
print(len(embedding_response["embeddings"][0]))

<class 'ollama._types.EmbedResponse'>
768


In [16]:
for chunk in chunks:
    response = ollama.embed(
        model="nomic-embed-text",
        input=chunk["text"]
    )
    
    chunk["embedding"] = response["embeddings"][0]

print("Embedded chunks:", len(chunks))
print("Embedding dimensions:", len(chunks[0]["embedding"]))

Embedded chunks: 57
Embedding dimensions: 768


### Build semantic search

In [17]:
import numpy as np

def cosine_similarity(vector_a, vector_b):
    vector_a = np.array(vector_a)
    vector_b = np.array(vector_b)

    return np.dot(vector_a, vector_b) / (
        np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    )

In [18]:
print(cosine_similarity([1, 0], [1, 0]))
print(cosine_similarity([1, 0], [0, 1]))

1.0
0.0


In [19]:
def retrieve_documents(question, top_k=3):
    
    # Create an embedding for the user's question
    response = ollama.embed(
        model="nomic-embed-text",
        input=question
    )
    
    question_embedding = response["embeddings"][0]
    
    # Compare the question with every document chunk
    scored_chunks = []
    
    for chunk in chunks:
        score = cosine_similarity(
            question_embedding,
            chunk["embedding"]
        )
        
        scored_chunks.append({
            "source": chunk["source"],
            "text": chunk["text"],
            "score": score
        })
    
    # Sort from most relevant to least relevant
    scored_chunks.sort(
        key=lambda x: x["score"],
        reverse=True
    )
    
    return scored_chunks[:top_k]

In [20]:
results = retrieve_documents(
    "What information should a supplier provide when goods are shipped?",
    top_k=3
)

for result in results:
    print("Source:", result["source"])
    print("Score:", round(result["score"], 4))
    print("Text:", result["text"])
    print("-" * 60)

Source: ..\documents\supplier\supplier_delivery_policy.txt
Score: 0.8828
Text: Suppliers should provide shipment and delivery confirmation information when goods are dispatched and delivered.
------------------------------------------------------------
Source: ..\documents\logistics\carrier_service_guidelines.txt
Score: 0.7442
Text: Shipment tracking information should be captured where available.
------------------------------------------------------------
Source: ..\documents\supplier\supplier_delivery_policy.txt
Score: 0.7244
Text: Suppliers are expected to deliver orders by the agreed delivery date specified in the purchase or logistics agreement.
------------------------------------------------------------


### Generate an answer from the retrieved context

In [21]:
question = "What information should a supplier provide when goods are shipped?"

results = retrieve_documents(question, top_k=3)

context = "\n\n".join(
    result["text"]
    for result in results
)

answer_prompt = f"""
You are a logistics data analyst.

Answer the user's question using only the information provided in the document context.

User question:
{question}

Document context:
{context}

Requirements:
- Use only the provided context.
- Do not invent information.
- Keep the answer concise and business-friendly.
- If the context does not contain enough information, say so.
"""

response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": answer_prompt
        }
    ]
)

print(response["message"]["content"])

According to the document, a supplier should provide:

1. Shipment and delivery confirmation information when goods are dispatched and delivered.
2. Shipment tracking information, where available.

However, the document does not specify what other information a supplier should provide when shipping goods.


In [26]:
def ask_document_question(user_question, top_k=3):
    
    # 1. Retrieve relevant document chunks
    results = retrieve_documents(
        user_question,
        top_k=top_k
    )
    
    # 2. Build source-aware context
    context_parts = []
    
    for result in results:
        context_parts.append(
            f"Source: {result['source']}\n"
            f"Content: {result['text']}"
        )
    
    context = "\n\n".join(context_parts)
    
    # 3. Ask the LLM to answer using the retrieved context
    answer_prompt = f"""
    You are a logistics data analyst answering a business question.

    Use ONLY the information explicitly stated in the document context.

    User question:
    {user_question}

    Document context:
    {context}

    Instructions:
    - Answer the question directly.
    - Preserve important details from the source text.
    - Do not omit relevant information that directly answers the question.
    - Do not combine separate requirements unless the documents clearly support doing so.
    - Do not invent information.
    - If multiple documents contain relevant information, distinguish them.
    - If the documents do not provide enough information, say that clearly.
    - Keep the answer concise.
    """

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": answer_prompt
            }
        ]
    )
    
    return {
        "question": user_question,
        "sources": results,
        "answer": response["message"]["content"].strip()
    }

In [23]:
response = ask_document_question(
    "What information should a supplier provide when goods are shipped?"
)

print("Question:", response["question"])
print()
print("Answer:", response["answer"])
print()
print("Sources:")

for source in response["sources"]:
    print(
        source["source"],
        "| Score:",
        round(source["score"], 4)
    )

Question: What information should a supplier provide when goods are shipped?

Answer: Based on the provided document context, a supplier should provide shipment confirmation information when goods are shipped.

Sources:
..\documents\supplier\supplier_delivery_policy.txt | Score: 0.8828
..\documents\logistics\carrier_service_guidelines.txt | Score: 0.7442
..\documents\supplier\supplier_delivery_policy.txt | Score: 0.7244


In [27]:
question = "What information should a supplier provide when goods are shipped?"

results = retrieve_documents(
    question,
    top_k=3
)

for i, result in enumerate(results, start=1):
    print(f"--- Result {i} ---")
    print("Source:", result["source"])
    print("Score:", round(result["score"], 4))
    print("Text:")
    print(result["text"])
    print()

--- Result 1 ---
Source: ..\documents\supplier\supplier_delivery_policy.txt
Score: 0.8828
Text:
Suppliers should provide shipment and delivery confirmation information when goods are dispatched and delivered.

--- Result 2 ---
Source: ..\documents\logistics\carrier_service_guidelines.txt
Score: 0.7442
Text:
Shipment tracking information should be captured where available.

--- Result 3 ---
Source: ..\documents\supplier\supplier_delivery_policy.txt
Score: 0.7244
Text:
Suppliers are expected to deliver orders by the agreed delivery date specified in the purchase or logistics agreement.



In [28]:
response = ask_document_question(
    "What information should a supplier provide when goods are shipped?"
)

print("Question:", response["question"])
print()
print("Answer:", response["answer"])
print()
print("Sources:")

for source in response["sources"]:
    print(
        source["source"],
        "| Score:",
        round(source["score"], 4)
    )

Question: What information should a supplier provide when goods are shipped?

Answer: According to the supplier delivery policy, suppliers should provide shipment and delivery confirmation information when goods are dispatched and delivered.

Sources:
..\documents\supplier\supplier_delivery_policy.txt | Score: 0.8828
..\documents\logistics\carrier_service_guidelines.txt | Score: 0.7442
..\documents\supplier\supplier_delivery_policy.txt | Score: 0.7244


In [29]:
response = ask_document_question(
    "Who is generally responsible for transportation under EXW?"
)

print("Question:", response["question"])
print()
print("Answer:", response["answer"])
print()
print("Sources:")

for source in response["sources"]:
    print(
        source["source"],
        "| Score:",
        round(source["score"], 4)
    )

Question: Who is generally responsible for transportation under EXW?

Answer: Under EXW (Ex Works), the buyer is generally responsible for arranging transportation from the agreed location.

Sources:
..\documents\logistics\incoterms_guidance.txt | Score: 0.6963
..\documents\logistics\incoterms_guidance.txt | Score: 0.6696
..\documents\logistics\incoterms_guidance.txt | Score: 0.6603


In [30]:
def ask_document_question(user_question, top_k=3):
    
    # 1. Retrieve relevant document chunks
    results = retrieve_documents(
        user_question,
        top_k=top_k
    )
    
    # 2. Build source-aware context
    context_parts = []
    
    for result in results:
        context_parts.append(
            f"Source: {result['source']}\n"
            f"Content: {result['text']}"
        )
    
    context = "\n\n".join(context_parts)
    
    # 3. Generate answer
    answer_prompt = f"""
    You are a logistics data analyst answering a business question.

    Use ONLY the information explicitly stated in the document context.

    User question:
    {user_question}

    Document context:
    {context}

    Instructions:
    - Answer the question directly.
    - Preserve important details from the source text.
    - Do not invent information.
    - If multiple documents contain relevant information, distinguish them.
    - If the documents do not provide enough information, say so clearly.
    - Keep the answer concise.
    """

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": answer_prompt
            }
        ]
    )
    
    # 4. Create clean source information
    sources = []
    
    for result in results:
        sources.append({
            "source": result["source"],
            "score": round(result["score"], 4),
            "text": result["text"]
        })
    
    # 5. Return structured result
    return {
        "question": user_question,
        "answer": response["message"]["content"].strip(),
        "sources": sources
    }

In [31]:
response = ask_document_question(
    "Who is generally responsible for transportation under EXW?"
)

print("Question:", response["question"])
print()
print("Answer:", response["answer"])
print()
print("Sources:")

for source in response["sources"]:
    print("Source:", source["source"])
    print("Score:", source["score"])
    print("Text:", source["text"])
    print()

Question: Who is generally responsible for transportation under EXW?

Answer: Under EXW, the buyer is generally responsible for arranging transportation from the agreed location.

Sources:
Source: ..\documents\logistics\incoterms_guidance.txt
Score: 0.6963
Text: The buyer generally takes responsibility for arranging transportation from the agreed location.

Source: ..\documents\logistics\incoterms_guidance.txt
Score: 0.6696
Text: Under DAP, the seller is generally responsible for arranging transportation to the named destination.

Source: ..\documents\logistics\incoterms_guidance.txt
Score: 0.6603
Text: Under DDP, the seller generally takes responsibility for transporting the goods to the named destination and handling import formalities and applicable duties.

